# Markus Barosh Nocturne – Colab

Colab ist der **Motor**, Nocturne das **Cockpit**: Du arbeitest in Nocturne, hier rechnet nur die Grafikkarte.

1. Oben **Laufzeit › Laufzeittyp ändern › GPU** (A100 für MiniMax H3, L4 für Bilder und Wan 2.2).
2. **Laufzeit › Alle ausführen** (Strg/⌘+F9). Google fragt einmal pro Sitzung nach dem Drive-Zugriff.
3. Beim ersten Öffnen fragt Google, ob du dem Notebook vertraust („Trotzdem ausführen“). Am Ende erscheint „Bereit – mit Nocturne verbinden“. Klicke den Knopf – oder „Code kopieren“, Nocturne erkennt ihn selbst.
4. Lass diesen Tab offen (im Hintergrund genügt). Bei Leerlauf gibt das Notebook die GPU selbst frei.

Hinweis: Kostenloses Colab erlaubt die Nutzung über eine Web-Oberfläche nicht. Nutze Colab Pro oder „Pay As You Go“ mit Guthaben.

In [ ]:
#@title 1 · Vorbereiten (ComfyUI, Google Drive)
DATEN = "/content/drive/MyDrive/ComfyUI-Data"  #@param {type:"string"}
LEERLAUF_MINUTEN = 30  #@param {type:"integer"}
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(os.path.join(DATEN, "models"), exist_ok=True)
if not os.path.isdir("/content/ComfyUI"):
    subprocess.run("git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI", shell=True, check=True)
    subprocess.run("pip install -q -r /content/ComfyUI/requirements.txt", shell=True, check=False)
# Modelle bleiben dauerhaft in deinem Drive; Ergebnisse nur kurz auf der Colab-Maschine (/content/ausgabe)
with open("/content/ComfyUI/extra_model_paths.yaml", "w") as f:
    f.write("nocturne:\n  base_path: " + os.path.join(DATEN, "models") + "\n" + "".join(f"  {o}: {o}/\n" for o in ["checkpoints", "diffusion_models", "unet", "loras", "vae", "text_encoders", "clip", "clip_vision", "upscale_models", "controlnet", "audio_encoders", "model_patches", "latent_upscale_models"]))
os.makedirs("/content/ausgabe", exist_ok=True)
print("Vorbereitet.")

In [ ]:
#@title 2 · Zugänge (optional)
#@markdown Welche Modelle Colab bereithält, wählst du in Nocturne (Fenster „Google Colab“). Nach dem Verbinden lädt Colab nur, was in deinem Drive noch fehlt.
#@markdown Für Modelle mit Anmeldung: links im Colab-Menü „Secrets“ (Schlüssel-Symbol) HF_TOKEN bzw. CIVITAI_TOKEN anlegen – optional.
import os
def geheim(name):
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return os.environ.get(name, "")
GEHEIM = {"HF": geheim("HF_TOKEN"), "CIVITAI": geheim("CIVITAI_TOKEN")}
print("Zugänge:", ", ".join(k for k, v in GEHEIM.items() if v) or "keine (nur frei verfügbare Modelle)")

In [ ]:
%%writefile /content/nocturne_bruecke.py
# Markus Barosh Nocturne – Brücke für Google Colab
# Schützt deine ComfyUI mit einem geheimen Schlüssel (ComfyUI selbst hat keine Anmeldung), stellt sie über einen
# Cloudflare-Tunnel bereit, meldet sich bei Nocturne an, löscht abgeholte Ergebnisse auf Wunsch und gibt die GPU
# nach längerer Pause frei, damit keine Recheneinheiten verbrennen.
# Läuft in einem Hintergrund-Thread des Notebooks. Nichts davon sendet Daten an Dritte außer dem Tunnel selbst.
# Ab Version 2: „Einrichten“ – Nocturne schickt nach dem Verbinden, was gebraucht wird (Erweiterungen, Modelle);
# die Brücke lädt nur, was im Drive noch fehlt (erst als .teil, dann umbenannt) und startet ComfyUI bei neuen
# Erweiterungen neu. Erlaubt sind nur Modelle von huggingface.co/civitai.com und Erweiterungen von github.com.
import asyncio
import base64
import hmac
import json
import os
import re
import secrets
import subprocess
import threading
import time
import urllib.request

COMFY = os.environ.get('NOCTURNE_COMFY', 'http://127.0.0.1:8188')
PORT = int(os.environ.get('NOCTURNE_PORT', '8189'))
AUSGABE = os.path.realpath(os.environ.get('NOCTURNE_AUSGABE', '/content/ausgabe'))
LEERLAUF_MIN = float(os.environ.get('NOCTURNE_LEERLAUF_MIN', '30'))
SCHLUESSEL = os.environ.get('NOCTURNE_SCHLUESSEL') or secrets.token_urlsafe(24)
VERSION = 2
MODELLE = os.path.realpath(os.environ.get('NOCTURNE_MODELLE', '/content/drive/MyDrive/ComfyUI-Data/models'))
COMFY_ORDNER = os.environ.get('NOCTURNE_COMFY_ORDNER', '/content/ComfyUI')
# Nur für Tests: ein zusätzlicher erlaubter Download-Ursprung (z. B. http://127.0.0.1:18290)
TEST_ERLAUBT = os.environ.get('NOCTURNE_TEST_ERLAUBT', '')
ORDNER_OK = {'checkpoints', 'diffusion_models', 'unet', 'loras', 'vae', 'text_encoders', 'clip', 'clip_vision', 'upscale_models',
             'controlnet', 'audio_encoders', 'model_patches', 'latent_upscale_models'}
ENDUNG_OK = re.compile(r'^[\w.\-+() ]{1,180}\.(safetensors|sft|ckpt|pt|pth|gguf|bin|onnx)$', re.I)
NODE_OK = re.compile(r'^https://github\.com/[A-Za-z0-9_.-]{1,80}/[A-Za-z0-9_.-]{1,100}?(\.git)?/?$')
# Tokens für Downloads (vom Notebook aus den Colab-Secrets gesetzt) – nur im Arbeitsspeicher, nie auf der Platte
GEHEIM = {}

_zustand = {'letzte': time.time(), 'tunnel': '', 'beendet': False}
_comfy = {'prozess': None, 'befehl': None, 'cwd': None, 'log': None}
_einrichtung = {'laeuft': False, 'schritt': '', 'nr': 0, 'von': 0, 'datei': '', 'bytes': 0, 'gesamt': 0, 'fertig': [], 'vorhanden': [], 'fehler': [], 'neustart': False, 'ende': 0}


def _gpu():
    try:  # zuerst ComfyUI fragen (welche Karte es tatsächlich nutzt), sonst nvidia-smi
        d = json.load(urllib.request.urlopen(COMFY + '/system_stats', timeout=5))
        g = (d.get('devices') or [{}])[0]
        if g.get('name'):
            return {'gpu': re.sub(r'^cuda:\d+\s*', '', str(g['name'])).split(' : ')[0], 'vramGB': round((g.get('vram_total') or 0) / 1e9)}
    except Exception:
        pass
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=5).stdout.strip()
        if out:
            name, mem = [x.strip() for x in out.split('\n')[0].split(',')]
            return {'gpu': name, 'vramGB': round(int(mem) / 1024)}
    except Exception:
        pass
    return {'gpu': '', 'vramGB': 0}


def _laufzeit_beenden():
    """Colab-Laufzeit freigeben (GPU zurückgeben). Außerhalb von Colab: nur den Server anhalten."""
    _zustand['beendet'] = True
    try:
        from google.colab import runtime  # type: ignore
        runtime.unassign()
    except Exception:
        os._exit(0)


def comfy_starten(befehl, cwd, log_pfad='/content/comfyui.log', warten=600):
    """ComfyUI starten (vom Notebook aufgerufen) und warten, bis es antwortet. Die Brücke merkt sich den Prozess,
    damit sie ComfyUI nach dem Installieren neuer Erweiterungen neu starten kann."""
    _comfy.update({'befehl': list(befehl), 'cwd': cwd, 'log': log_pfad})
    log = open(log_pfad, 'a')
    _comfy['prozess'] = subprocess.Popen(list(befehl), cwd=cwd, stdout=log, stderr=subprocess.STDOUT)
    ende = time.time() + warten
    while time.time() < ende:
        try:
            urllib.request.urlopen(COMFY + '/system_stats', timeout=2)
            return True
        except Exception:
            if _comfy['prozess'].poll() is not None:
                return False
            time.sleep(2)
    return False


def comfy_neustart():
    p = _comfy.get('prozess')
    if not _comfy.get('befehl'):
        return False
    if p and p.poll() is None:
        p.terminate()
        try:
            p.wait(timeout=30)
        except Exception:
            p.kill()
    return comfy_starten(_comfy['befehl'], _comfy['cwd'], _comfy['log'])


def _url_ok(url):
    u = str(url or '')
    if TEST_ERLAUBT and u.startswith(TEST_ERLAUBT.rstrip('/') + '/'):
        return True
    # Hugging Face, Civitai und die offiziellen Real-ESRGAN-Veröffentlichungen (Hochskalieren)
    return bool(re.match(r'^https://(huggingface\.co|civitai\.com)/', u) or u.startswith('https://github.com/xinntao/Real-ESRGAN/releases/download/'))


def _laden(url, ziel):
    """Eine Datei laden: erst als .teil, erst am Ende umbenennen – ein Abbruch hinterlässt keine halbe Modelldatei."""
    kopf = {'User-Agent': 'Nocturne-Colab/2'}
    quelle = url
    if 'huggingface.co' in url and GEHEIM.get('HF'):
        kopf['Authorization'] = 'Bearer ' + GEHEIM['HF']
    if 'civitai.com' in url and GEHEIM.get('CIVITAI'):
        quelle = url + ('&' if '?' in url else '?') + 'token=' + GEHEIM['CIVITAI']
    teil = ziel + '.teil'
    os.makedirs(os.path.dirname(ziel), exist_ok=True)
    with urllib.request.urlopen(urllib.request.Request(quelle, headers=kopf), timeout=60) as r, open(teil, 'wb') as f:
        _einrichtung['gesamt'] = int(r.headers.get('Content-Length') or 0)
        _einrichtung['bytes'] = 0
        while True:
            stueck = r.read(1 << 20)
            if not stueck:
                break
            f.write(stueck)
            _einrichtung['bytes'] += len(stueck)
            _zustand['letzte'] = time.time()
    if os.path.getsize(teil) < 100000:
        os.remove(teil)
        raise RuntimeError('Download zu klein – braucht das Modell eine Anmeldung (Colab-Secrets HF_TOKEN / CIVITAI_TOKEN)?')
    os.replace(teil, ziel)


def _einrichten(auftrag):
    e = _einrichtung
    try:
        nodes = [n for n in (auftrag.get('nodes') or []) if isinstance(n, str)]
        modelle = [m for m in (auftrag.get('modelle') or []) if isinstance(m, (list, tuple)) and len(m) == 3]
        e.update({'laeuft': True, 'schritt': 'erweiterungen', 'nr': 0, 'von': len(nodes) + len(modelle), 'fertig': [], 'vorhanden': [], 'fehler': [], 'neustart': False})
        neu = 0
        for url in nodes:
            e['nr'] += 1
            if not NODE_OK.match(url):
                e['fehler'].append({'was': url, 'grund': 'nicht erlaubt (nur github.com)'})
                continue
            name = url.rstrip('/').split('/')[-1].replace('.git', '')
            ziel = os.path.join(COMFY_ORDNER, 'custom_nodes', name)
            e['datei'] = name
            if os.path.isdir(ziel):
                e['vorhanden'].append(name)
                continue
            r = subprocess.run(['git', 'clone', '--depth', '1', url, ziel], capture_output=True, text=True)
            if r.returncode != 0:
                e['fehler'].append({'was': name, 'grund': (r.stderr or 'git clone fehlgeschlagen')[-300:]})
                continue
            req = os.path.join(ziel, 'requirements.txt')
            if os.path.isfile(req):
                subprocess.run(['pip', 'install', '-q', '-r', req], capture_output=True, text=True)
            e['fertig'].append(name)
            neu += 1
        e['schritt'] = 'modelle'
        gesehen = set()
        for ordner, datei, url in modelle:
            e['nr'] += 1
            ordner, datei, url = str(ordner), str(datei), str(url)
            if (ordner, datei) in gesehen:
                continue
            gesehen.add((ordner, datei))
            e['datei'] = datei
            if ordner not in ORDNER_OK or not ENDUNG_OK.match(datei) or '/' in datei or '..' in datei:
                e['fehler'].append({'was': datei, 'grund': 'ungültiger Ordner oder Dateiname'})
                continue
            if not _url_ok(url):
                e['fehler'].append({'was': datei, 'grund': 'Quelle nicht erlaubt (nur huggingface.co und civitai.com)'})
                continue
            ziel = os.path.join(MODELLE, ordner, datei)
            if not os.path.realpath(ziel).startswith(MODELLE + os.sep):
                e['fehler'].append({'was': datei, 'grund': 'außerhalb des Modellordners'})
                continue
            if os.path.isfile(ziel):
                e['vorhanden'].append(datei)
                continue
            try:
                _laden(url, ziel)
                e['fertig'].append(datei)
            except Exception as x:
                e['fehler'].append({'was': datei, 'grund': str(x)[:300]})
        if neu:
            e['schritt'] = 'neustart'
            e['neustart'] = bool(comfy_neustart())
        e['schritt'] = 'fertig'
    except Exception as x:
        e['fehler'].append({'was': 'einrichten', 'grund': str(x)[:300]})
        e['schritt'] = 'fertig'
    finally:
        e['laeuft'] = False
        e['ende'] = time.time()


def _schluessel_ok(request):
    gegeben = request.headers.get('X-Nocturne-Schluessel') or request.query.get('nocturne_schluessel') or ''
    return hmac.compare_digest(gegeben.encode(), SCHLUESSEL.encode())


def _app():
    from aiohttp import web, ClientSession, ClientTimeout, WSMsgType

    async def status(_request):
        return web.json_response(dict({'ok': True, 'version': VERSION, 'leerlaufMin': LEERLAUF_MIN, 'tunnel': _zustand['tunnel']}, **_gpu()))

    async def loeschen(request):
        d = await request.json()
        name = os.path.basename(str(d.get('filename', '')))
        unter = str(d.get('subfolder', '')).strip('/')
        if not name or '..' in unter or str(d.get('type', 'output')) != 'output':
            return web.json_response({'ok': False, 'fehler': 'ungültig'}, status=400)
        ziel = os.path.realpath(os.path.join(AUSGABE, unter, name))
        if not ziel.startswith(AUSGABE + os.sep):
            return web.json_response({'ok': False, 'fehler': 'außerhalb'}, status=400)
        try:
            os.remove(ziel)
            return web.json_response({'ok': True})
        except FileNotFoundError:
            return web.json_response({'ok': True, 'schon_weg': True})

    async def beenden(_request):
        threading.Timer(1.0, _laufzeit_beenden).start()
        return web.json_response({'ok': True})

    async def weiter(request):
        if not _schluessel_ok(request):
            return web.Response(status=401, text='Nocturne-Schlüssel fehlt oder stimmt nicht.')
        _zustand['letzte'] = time.time()
        pfad = request.rel_url.path
        if pfad == '/nocturne/status':
            return await status(request)
        if pfad == '/nocturne/loeschen' and request.method == 'POST':
            return await loeschen(request)
        if pfad == '/nocturne/beenden' and request.method == 'POST':
            return await beenden(request)
        if pfad == '/nocturne/einrichten':
            if request.method == 'POST':
                if _einrichtung['laeuft']:
                    return web.json_response(dict(_einrichtung, ok=False, fehlerText='läuft schon'), status=409)
                auftrag = await request.json()
                threading.Thread(target=_einrichten, args=(auftrag if isinstance(auftrag, dict) else {},), daemon=True).start()
                await asyncio.sleep(0.05)
                return web.json_response({'ok': True})
            return web.json_response(dict(_einrichtung, ok=True))
        query = [(k, v) for k, v in request.query.items() if k != 'nocturne_schluessel']
        if request.headers.get('Upgrade', '').lower() == 'websocket':
            ws_client = web.WebSocketResponse()
            await ws_client.prepare(request)
            async with ClientSession() as s:
                async with s.ws_connect(COMFY.replace('http', 'ws', 1) + pfad, params=query) as ws_comfy:
                    async def von_comfy():
                        async for m in ws_comfy:
                            _zustand['letzte'] = time.time()
                            if m.type == WSMsgType.TEXT:
                                await ws_client.send_str(m.data)
                            elif m.type == WSMsgType.BINARY:
                                await ws_client.send_bytes(m.data)
                    aufgabe = asyncio.ensure_future(von_comfy())
                    async for m in ws_client:
                        if m.type == WSMsgType.TEXT:
                            await ws_comfy.send_str(m.data)
                    aufgabe.cancel()
            return ws_client
        kopf = {k: v for k, v in request.headers.items() if k.lower() not in ('host', 'x-nocturne-schluessel', 'content-length', 'accept-encoding')}
        daten = await request.read()
        async with ClientSession(timeout=ClientTimeout(total=None, sock_read=600)) as s:
            async with s.request(request.method, COMFY + pfad, params=query, headers=kopf, data=daten if daten else None) as r:
                antwort = web.StreamResponse(status=r.status, headers={k: v for k, v in r.headers.items() if k.lower() in ('content-type', 'content-disposition', 'cache-control')})
                await antwort.prepare(request)
                async for teil in r.content.iter_chunked(1 << 16):
                    await antwort.write(teil)
                await antwort.write_eof()
                return antwort

    app = web.Application(client_max_size=1024 ** 3)
    app.router.add_route('*', '/{pfad:.*}', weiter)
    return app


def _server_starten():
    from aiohttp import web
    schleife = asyncio.new_event_loop()
    asyncio.set_event_loop(schleife)
    runner = web.AppRunner(_app())
    schleife.run_until_complete(runner.setup())
    schleife.run_until_complete(web.TCPSite(runner, '127.0.0.1', PORT).start())
    schleife.run_forever()


def _leerlauf_waechter():
    while not _zustand['beendet']:
        time.sleep(max(0.5, min(30.0, LEERLAUF_MIN * 15)))
        if LEERLAUF_MIN <= 0:
            continue
        if time.time() - _zustand['letzte'] < LEERLAUF_MIN * 60:
            continue
        try:  # nur freigeben, wenn ComfyUI nichts mehr rechnet
            q = json.load(urllib.request.urlopen(COMFY + '/queue', timeout=5))
            if q.get('queue_running') or q.get('queue_pending'):
                _zustand['letzte'] = time.time()
                continue
        except Exception:
            pass
        print(f'Nocturne: {LEERLAUF_MIN:.0f} Minuten ohne Auftrag – die GPU wird freigegeben.')
        _laufzeit_beenden()


def verbindungscode(url):
    roh = json.dumps({'url': url, 'schluessel': SCHLUESSEL, 'v': VERSION}, separators=(',', ':')).encode()
    return base64.urlsafe_b64encode(roh).decode().rstrip('=')


def tunnel_starten(cloudflared='/content/cloudflared'):
    if not os.path.isfile(cloudflared):
        urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
        os.chmod(cloudflared, 0o755)
    p = subprocess.Popen([cloudflared, 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{PORT}'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    ende = time.time() + 90
    for zeile in p.stdout:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', zeile)
        if m:
            _zustand['tunnel'] = m.group(0)
            threading.Thread(target=lambda: [None for _ in p.stdout], daemon=True).start()
            return m.group(0)
        if time.time() > ende:
            break
    raise RuntimeError('Der Tunnel ist nicht zustande gekommen. Bitte die Zelle erneut ausführen.')


def starten(mit_tunnel=True):
    threading.Thread(target=_server_starten, daemon=True).start()
    threading.Thread(target=_leerlauf_waechter, daemon=True).start()
    time.sleep(1.0)
    url = tunnel_starten() if mit_tunnel else f'http://127.0.0.1:{PORT}'
    return url, verbindungscode(url)


def anzeigen(url, code):
    """Verbindungsfeld im Notebook: Knopf (öffnet Nocturne) und Code zum Kopieren (Nocturne erkennt ihn automatisch)."""
    from IPython.display import HTML, display  # type: ignore
    link = 'nocturne://verbinden?code=' + code
    display(HTML(f'''
<div style="font-family:system-ui,sans-serif;border:1px solid #a8701b;border-radius:12px;padding:16px;background:#131c2e;color:#ece6d6;max-width:640px">
  <div style="font-size:12px;color:#ffc565;font-weight:600;letter-spacing:.03em">Markus Barosh Nocturne</div>
  <div style="font-size:20px;font-weight:600;margin:4px 0 10px">Bereit – mit Nocturne verbinden</div>
  <a href="{link}" style="display:inline-block;background:#f2a93b;color:#0a101c;padding:9px 16px;border-radius:8px;font-weight:600;text-decoration:none">Mit Nocturne verbinden</a>
  <button onclick="navigator.clipboard.writeText('NOCTURNE-COLAB:{code}').then(()=>{{this.textContent='Kopiert – Nocturne verbindet sich'}})" style="margin-left:8px;background:transparent;color:#ece6d6;border:1px solid #3a4b72;padding:8px 14px;border-radius:8px;cursor:pointer">Code kopieren</button>
  <div style="font-size:12.5px;color:#bdb6a5;margin-top:12px;line-height:1.5">Falls der Knopf nichts öffnet: „Code kopieren“ – Nocturne erkennt den Code in der Zwischenablage automatisch, solange es auf Colab wartet.<br>Lass diesen Tab offen (im Hintergrund genügt). Nach {LEERLAUF_MIN:.0f} Minuten ohne Auftrag gibt das Notebook die GPU frei.</div>
  <details style="margin-top:8px;font-size:12px;color:#8189a0"><summary>Code anzeigen</summary><code style="word-break:break-all">NOCTURNE-COLAB:{code}</code></details>
</div>'''))


In [ ]:
#@title 3 · Starten und mit Nocturne verbinden
import os, subprocess, sys, time, urllib.request
os.environ["NOCTURNE_LEERLAUF_MIN"] = str(LEERLAUF_MINUTEN)
os.environ["NOCTURNE_AUSGABE"] = "/content/ausgabe"
os.environ["NOCTURNE_MODELLE"] = os.path.join(DATEN, "models")
sys.path.insert(0, "/content")
import nocturne_bruecke as nb
nb.GEHEIM.update(GEHEIM)  # Zugänge nur im Arbeitsspeicher
open("/content/comfyui.log", "w").close()
if not nb.comfy_starten([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--output-directory", "/content/ausgabe", "--preview-method", "none"], "/content/ComfyUI"):
    print(open("/content/comfyui.log").read()[-3000:]); raise SystemExit("ComfyUI ist nicht gestartet – siehe Protokoll oben.")
url, code = nb.starten()
nb.anzeigen(url, code)

In [ ]:
#@title GPU freigeben (wenn du fertig bist – oder in Nocturne „Colab beenden“)
from google.colab import runtime
runtime.unassign()